In [45]:
import geopandas as gpd
import pandas as pd
from shapely.ops import transform
import matplotlib.pyplot as plt
import numpy as np
import shapely
from shapely.geometry import Polygon
import pyvista as pv
import os
import cv2

In [46]:
import pandas as pd

input_file = r"C:/Users/baile/OneDrive/Desktop/Research_Orientation/AT-Export.csv"
output_file = r"C:/Users/baile/OneDrive/Desktop/underpass-detection-experiments/3DBAG_underpass_heights/data/rotterdam_camera_parameters.csv"


columns_to_rename = {"X Calculated": "X","Y Calculated": "Y","Altitude Calculated": "Z",
                     "Omega Calculated": "omega","Kappa Calculated": "kappa","Phi Calculated": "phi",
                     "PPoint X Calculated": "cx","PPoint Y Calculated": "cy",}


# read csv
df = pd.read_csv(input_file,sep=",",encoding="utf-16",skiprows=1)
df.columns = df.columns.str.strip()


df_selected = df[list(columns_to_rename.keys())].copy()
df_selected.rename(columns=columns_to_rename,inplace=True)

# Calculate fx and fy
df_selected["fx"] = (pd.to_numeric(df["Focal Length Calculated"], errors="coerce")/ pd.to_numeric(df["Pixel Size Input"], errors="coerce"))

df_selected["fy"] = (pd.to_numeric(df["Focal Length Calculated"], errors="coerce")/ pd.to_numeric(df["Pixel Size Input"], errors="coerce"))

# save new CSV
df_selected.to_csv(output_file,index=False)

print("Done!")
print(f"Saved to: {output_file}")

Done!
Saved to: C:/Users/baile/OneDrive/Desktop/underpass-detection-experiments/3DBAG_underpass_heights/data/rotterdam_camera_parameters.csv


In [48]:
import os
import geopandas as gpd
import pandas as pd
import pyogrio
PROJECT_ROOT = r"C:/Users/baile/OneDrive/Desktop/underpass-detection-experiments/3DBAG_underpass_heights"
my_footprint_path = os.path.join(PROJECT_ROOT, r"data/rotterdam_camera_parameters.csv")
rott_footprint = pd.read_csv(my_footprint_path)
rott_footprint.columns = rott_footprint.columns.str.strip()

# 2. Read Juan's camera parameters CSV
juan_camera_para_path = os.path.join(PROJECT_ROOT, r"data/oblique_images/camera_parameters.txt")
df_juan_camera_para = pd.read_csv(juan_camera_para_path, sep='\t', dtype={'image_id': str})
df_juan_camera_para.columns = df_juan_camera_para.columns.str.strip()

# Target comparison columns
compare_cols = ['fx', 'fy', 'omega', 'phi', 'kappa']

# Safely filter columns available in both datasets
my_available_cols = [c for c in compare_cols if c in rott_footprint.columns]
juan_available_cols = [c for c in compare_cols if c in df_juan_camera_para.columns]

# Print outputs
print("My Footprint Columns")
print(rott_footprint.columns.tolist())

print("\nMy Footprint Data (Head 5)")
print(rott_footprint[my_available_cols].head(5))

print("\nJuan Camera Parameters Data (Head 5)")
print(df_juan_camera_para[juan_available_cols].head(5))


My Footprint Columns
['X', 'Y', 'Z', 'omega', 'kappa', 'phi', 'cx', 'cy', 'fx', 'fy']

My Footprint Data (Head 5)
             fx            fy      omega       phi      kappa
0  21505.497194  21505.497194 -45.388552  0.078474 -90.062265
1  21505.497194  21505.497194 -44.752380  0.095639 -89.713862
2  21505.497194  21505.497194 -45.017590  0.094626 -90.049544
3  21505.497194  21505.497194 -45.230290  0.384170 -89.947904
4  21505.497194  21505.497194 -45.006232  0.376946 -89.796864

Juan Camera Parameters Data (Head 5)
             fx            fy  omega    phi   kappa
0  28831.117021  28831.117021  12.13  43.97  107.63
1  28831.117021  28831.117021  12.51  43.71  107.11
2  28836.941489  28836.941489  12.56  43.44  107.51
3  28831.117021  28831.117021  15.22 -41.53 -112.82
4  28831.117021  28831.117021  11.38  44.05  105.87


In [ ]:
import geopandas as gpd

# 1. Load the GeoJSON file
gdf = gpd.read_file(r"C:/Users/baile/OneDrive/Desktop/Research_Orientation/rotterdam_footprints_raw.geojson")

# 2. Rename existing columns (pass a dictionary of {'old_name': 'new_name'})
gdf = gdf.rename(columns={
    "PPoint X Calculated": "cx",
    "PPoint Y Calculated": "cy",
    "Y Calculated": "X",
    "X Calculated": "Y",
    "Altitude Calculated": "Z",
    "Omega Calculated": "omega",
    "Kappa Calculated": "kappa",
    "Phi Calculated": "phi",})

# 2. Divide column_a by column_b (safely handling zeros)
gdf["fx"] = gdf["Focal Length Calculated"] / gdf["Pixel Size Input"]
gdf["fy"] = gdf["Focal Length Calculated"] / gdf["Pixel Size Input"]

# 4. Save back to a GeoJSON file
gdf.to_file("C:/Users/baile/OneDrive/Desktop/Research_Orientation/rotterdam_footprints_renamed.geojson", driver="GeoJSON")

KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython.message.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq\\backend\\cython\\checkrc.pxd", line 13, in zmq.backend.cython.checkrc._check_rc
KeyboardInterrupt: 


In [5]:
80.830002/0.00376
# SHOULD BE THIS VALUE FOR FOCAL LENGTH!!!

21497.340957446806

In [49]:
# CHECKING IF ALL KAPPA PHI OMEGA ARE IN DEGREES
print(rott_footprint['omega'].head(3))
print(rott_footprint['kappa'].head(3))
# rott_footprint['phi'] = rott_footprint_raw['Phi Input']
print(rott_footprint['phi'].head(3))

0   -45.388552
1   -44.752380
2   -45.017590
Name: omega, dtype: float64
0   -90.062265
1   -89.713862
2   -90.049544
Name: kappa, dtype: float64
0    0.078474
1    0.095639
2    0.094626
Name: phi, dtype: float64


In [76]:
# IN CASE OF DOUBLE TRANSFORMATION ACCIDENTALLY HAPPENING
import pandas as pd
import geopandas as gpd

footprint_path = "/Users/belinaileen/Desktop/aileen_underpass-detection/3DBAG_underpass_heights/data/oblique_images/rotterdam_footprints_renamed.geojson"
camera_path = "/Users/belinaileen/Desktop/aileen_underpass-detection/3DBAG_underpass_heights/output/rotterdam_camera_parameters.csv"

gdf_footprint = gpd.read_file(footprint_path)
df_camera = pd.read_csv(camera_path)

# Reverse the extra division by multiplying once
gdf_footprint['fx'] = gdf_footprint['fx'] * gdf_footprint['Pixel Size Input']
gdf_footprint['fy'] = gdf_footprint['fy'] * gdf_footprint['Pixel Size Input']

df_camera['fx'] = df_camera['fx'] * gdf_footprint['Pixel Size Input']
df_camera['fy'] = df_camera['fy'] * gdf_footprint['Pixel Size Input']

gdf_footprint.to_file(footprint_path, driver="GeoJSON")
df_camera.to_csv(camera_path, index=False)

print("Values successfully restored!")

Values successfully restored!


In [80]:
import os
import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = (
    "/Users/belinaileen/Desktop/aileen_underpass-detection/3DBAG_underpass_heights"
)
camera_parameters_path = os.path.join(
    PROJECT_ROOT, "output/rotterdam_camera_parameters.csv"
)
footprint_geojson_path = os.path.join(
    PROJECT_ROOT, "data/oblique_images/rotterdam_footprints_renamed.geojson"
)

gdf_footprint = gpd.read_file(footprint_geojson_path)
df_camera_csv = pd.read_csv(camera_parameters_path)

# Convert from double-degree back to degrees by applying deg2rad once
if "kappa" in gdf_footprint.columns and "omega" in gdf_footprint.columns:
    gdf_footprint["kappa"] = np.deg2rad(gdf_footprint["kappa"])
    gdf_footprint["omega"] = np.deg2rad(gdf_footprint["omega"])

if "kappa" in df_camera_csv.columns and "omega" in df_camera_csv.columns:
    df_camera_csv["kappa"] = np.deg2rad(df_camera_csv["kappa"])
    df_camera_csv["omega"] = np.deg2rad(df_camera_csv["omega"])

# Save corrected files back to disk
gdf_footprint.to_file(footprint_geojson_path, driver="GeoJSON")
df_camera_csv.to_csv(camera_parameters_path, index=False)

print("kappa and omega successfully restored to degrees!")

kappa and omega successfully restored to degrees!


In [85]:
print(df_camera_csv["omega"].head(5))
print(df_camera_csv["phi"].head(5))
print(df_camera_csv["kappa"].head(5))

0    23.107387
1     2.990840
2    -5.483206
3    57.605173
4     4.847223
Name: omega, dtype: float64
0    44.417198
1    45.103401
2    45.153702
3    43.948299
4    45.155399
Name: phi, dtype: float64
0   -15.962604
1   -17.521049
2   -15.876661
3   -10.783066
4   -18.913337
Name: kappa, dtype: float64


In [84]:

# Juan footprints
PROJECT_ROOT = "/Users/belinaileen/Desktop/aileen_underpass-detection/3DBAG_underpass_heights"
juan_camera_para = os.path.join(PROJECT_ROOT, "data/oblique_images/camera_parameters.txt")
juan_footprint = os.path.join(PROJECT_ROOT, "data/oblique_images/image_footprints.geojson")
df_juan_camera_para = pd.read_csv(juan_camera_para, sep='\t', dtype={'image_id': str})
gdf_juan_footprints = gpd.read_file(juan_footprint)
df_juan_camera_para.columns = df_juan_camera_para.columns.str.strip()
#print(df_juan_camera_para.head(10))
#print(gdf_juan_footprints.head(10))
print(df_juan_camera_para["omega"].head(5))
print(df_juan_camera_para["phi"].head(5))
print(df_juan_camera_para["kappa"].head(5))

0    12.13
1    12.51
2    12.56
3    15.22
4    11.38
Name: omega, dtype: float64
0    43.97
1    43.71
2    43.44
3   -41.53
4    44.05
Name: phi, dtype: float64
0    107.63
1    107.11
2    107.51
3   -112.82
4    105.87
Name: kappa, dtype: float64


In [23]:
from PIL import Image
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.transform import from_gcps

# 1. Define your inputs
# input_jpg = "/Users/belinaileen/.Trash/aileen_underpass-detection/3DBAG_underpass_heights/rotterdam_sample/oblique_left/217001859_0039_03_0116_P00_01.jpg"
input_jpg = "/Users/belinaileen/Desktop/rotterdam_sample/217001859_0039_03_0116_P00_01.jpg"
output_tif = "/Users/belinaileen/Desktop/217001859_0039_03_0116_P00_01_geocoded.tif"

crs_code = "EPSG:28992" 


# 93957.390965033817338, 439259.775083448155783, 0.0 ], 
# [ 93864.900400222832104, 438674.295225750713143, 0.0 ], 
# [ 93361.496539588901214, 438672.921533675282262, 0.0 ], 
# [ 93268.539689129407634, 439256.353740151389502, 0.0 ],
# [ 93957.390965033817338, 439259.775083448155783, 0.0 ] ] ] } },
poly_coords = [
    (93957.391, 439259.775), # Corner 1
    (93864.900, 438674.295), # Corner 2
    (93361.497, 438672.922), # Corner 3
    (93268.540, 439256.354)  # Corner 4
]

# 2. Get image pixel dimensions
with Image.open(input_jpg) as img:
    width, height = img.size

gcps = [
    GroundControlPoint(0, width, poly_coords[0][0], poly_coords[0][1]),       # Top-Right
    GroundControlPoint(height, width, poly_coords[1][0], poly_coords[1][1]),  # Bottom-Right
    GroundControlPoint(height, 0, poly_coords[2][0], poly_coords[2][1]),      # Bottom-Left
    GroundControlPoint(0, 0, poly_coords[3][0], poly_coords[3][1])            # Top-Left
]

transform = from_gcps(gcps)

with rasterio.open(input_jpg) as src:
    data = src.read()
    kwargs = src.meta.copy()
    
    kwargs.update({
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'transform': transform,
        'crs': crs_code
    })

    with rasterio.open(output_tif, 'w', **kwargs) as dst:
        dst.write(data)

print(f"GeoTIFF successfully created: {output_tif}")


GeoTIFF successfully created: /Users/belinaileen/Desktop/217001447_0037_01_0174_P00_01_geocoded.tif


In [ ]:
# example 2

input_jpg = "/Users/belinaileen/Desktop/rotterdam_sample/oblique_left/217001447_0037_01_0174_P00_01.jpg"
output_tif = "/Users/belinaileen/Desktop/217001447_0037_01_0174_P00_01_geocoded.tif"

crs_code = "EPSG:28992" 

# Extracted unique corners from your Polygon geometry (X, Y)
# 94179.460817126557231, 440393.598395447188523, 0.0 ], 
# [ 94077.837546463371837, 439766.688744171638973, 0.0 ], 
# [ 93559.478306596967741, 439765.056886383565143, 0.0 ], 
# [ 93454.9994477492437, 440390.721339175594039, 0.0 ], 
# [ 94179.460817126557231, 440393.598395447188523, 0.0 ] ] ] } },

poly_coords = [
    (94179.460817126557231, 440393.598395447188523), # Corner 1
    (94077.837546463371837, 439766.688744171638973), # Corner 2
    (93559.478306596967741, 439765.056886383565143), # Corner 3
    (93454.9994477492437, 440390.721339175594039)  # Corner 4
]
# rotate + 90 degrees to match the image orientation

with Image.open(input_jpg) as img:
    width, height = img.size

gcps = [
    GroundControlPoint(0, width, poly_coords[0][0], poly_coords[0][1]),       # Top-Right
    GroundControlPoint(height, width, poly_coords[1][0], poly_coords[1][1]),  # Bottom-Right
    GroundControlPoint(height, 0, poly_coords[2][0], poly_coords[2][1]),      # Bottom-Left
    GroundControlPoint(0, 0, poly_coords[3][0], poly_coords[3][1])]        # Top-Left


transform = from_gcps(gcps)

with rasterio.open(input_jpg) as src:
    data = src.read()
    kwargs = src.meta.copy()
    
    kwargs.update({
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'transform': transform,
        'crs': crs_code
    })

    with rasterio.open(output_tif, 'w', **kwargs) as dst:
        dst.write(data)

print(f"GeoTIFF successfully created: {output_tif}")


In [25]:
from PIL import Image
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.transform import from_gcps

input_jpg = "/Users/belinaileen/Desktop/rotterdam_sample/oblique_left/217001447_0037_01_0174_P00_01.jpg"
output_tif = "/Users/belinaileen/Desktop/217001447_0037_01_0174_P00_01_geocoded.tif"
crs_code = "EPSG:28992" 

poly_coords = [
    (94179.460817126557231, 440393.598395447188523), # Corner 1
    (94077.837546463371837, 439766.688744171638973), # Corner 2
    (93559.478306596967741, 439765.056886383565143), # Corner 3
    (93454.9994477492437, 440390.721339175594039)    # Corner 4
]

with Image.open(input_jpg) as img:
    width, height = img.size

# Shifting the pixel positions by +90 degrees (counter-clockwise)
gcps = [
    # Image Bottom-Right maps to Corner 1 (Top-Right geographic)
    GroundControlPoint(row=height, col=width, x=poly_coords[0][0], y=poly_coords[0][1]),       
    # Image Bottom-Left maps to Corner 2 (Bottom-Right geographic)
    GroundControlPoint(row=height, col=0,     x=poly_coords[1][0], y=poly_coords[1][1]),  
    # Image Top-Left maps to Corner 3 (Bottom-Left geographic)
    GroundControlPoint(row=0,      col=0,     x=poly_coords[2][0], y=poly_coords[2][1]),      
    # Image Top-Right maps to Corner 4 (Top-Left geographic)
    GroundControlPoint(row=0,      col=width, x=poly_coords[3][0], y=poly_coords[3][1])        
]

transform = from_gcps(gcps)

with rasterio.open(input_jpg) as src:
    data = src.read()
    kwargs = src.meta.copy()
    
    kwargs.update({
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'transform': transform,
        'crs': crs_code
    })

    with rasterio.open(output_tif, 'w', **kwargs) as dst:
        dst.write(data)

print(f"GeoTIFF successfully created: {output_tif}")


GeoTIFF successfully created: /Users/belinaileen/Desktop/217001447_0037_01_0174_P00_01_geocoded.tif


In [4]:
# example 3
from PIL import Image
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.transform import from_gcps

input_jpg = "/Users/belinaileen/Desktop/178001098_0036_01_0175_P00_01.jpg"
output_tif = "/Users/belinaileen/Desktop/178001098_0036_01_0175_P00_01_geocoded.tif"

crs_code = "EPSG:28992" 
'''
Try with these coordinates instead and replace the poly_coords
ULx	ULy	URx	URy	LLx	LLy	LRx	LRy
92139.81106793730 439536.05514660600	92165.22394669530	440075.03633848600	92814.86250525320	439417.95254586400	92890.85119572840	440201.4070365320
'''

poly_coords = [
    (92884.986046008445555, 439330.653237799764611), # Corner 1
    (92884.986046008445555, 439330.653237799764611), # Corner 2
    (93591.340439100502408, 440003.455825520155486), # Corner 3
    (93595.418613509114948, 439459.024791435280349)  # Corner 4
]
with Image.open(input_jpg) as img:
    width, height = img.size

gcps = [
    GroundControlPoint(0, width, poly_coords[0][0], poly_coords[0][1]),       # Top-Right
    GroundControlPoint(height, width, poly_coords[1][0], poly_coords[1][1]),  # Bottom-Right
    GroundControlPoint(height, 0, poly_coords[2][0], poly_coords[2][1]),      # Bottom-Left
    GroundControlPoint(0, 0, poly_coords[3][0], poly_coords[3][1])]        # Top-Left


transform = from_gcps(gcps)

with rasterio.open(input_jpg) as src:
    data = src.read()
    kwargs = src.meta.copy()
    
    kwargs.update({
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'transform': transform,
        'crs': crs_code
    })

    with rasterio.open(output_tif, 'w', **kwargs) as dst:
        dst.write(data)

print(f"GeoTIFF successfully created: {output_tif}")


/usr/local/Caskroom/miniconda/base/envs/geo/lib/python3.12/site-packages/PIL/Image.py:3578: DecompressionBombWarning: Image size (151301008 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/Caskroom/miniconda/base/envs/geo/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


GeoTIFF successfully created: /Users/belinaileen/Desktop/178001098_0036_01_0175_P00_01_geocoded.tif


In [19]:

'''
Parameters and intrinsic and extrinsic parameters
Name Input: 199001111_0036_01_0162_P00_01.jpg
id	ULx	ULy	URx	URy	LLx	LLy	LRx	LRy	GSD	ShotOrient	Inpho_11	Inpho_12	Inpho_13	Inpho_21	Inpho_22	Inpho_23	Inpho_31	Inpho_32	Inpho_33	Matrix_11	Matrix_12	Matrix_13	Matrix_21	Matrix_22	Matrix_23	Matrix_31	Matrix_32	Matrix_33	CAM	Collection Input	Name Input	Mean median error Calculated	Focal Length Calculated	Focal Length Input	Pixel Size Input	Median Error Calculated	PPoint X Calculated	PPoint X Input	PPoint Y Calculated	PPoint Y Input	K1 Calculated	K1 Input	K2 Calculated	K2 Input	K3 Calculated	K3 Input	P1 Calculated	P1 Input	P2 Calculated	P2 Input	X Calculated	X Input	Y Calculated	Y Input	Altitude Calculated	Altitude Input	Omega Calculated	Omega Input	Phi Calculated	Phi Input	Kappa Calculated	Kappa Input	Yaw Calculated	Yaw Input	Pitch Calculated	Pitch Input	Roll Calculated	Roll Input	Path Input	Width Input	Height Input	Camera system ID	Name (u)(r)	Type (r)	Width (px)(r)	Height (px)(r)	Width (mm)(r)	Height (mm)(r)	Focal Length (mm)(r)	Serial (u)(r)	PPX	PPY	Image_Z_AxisRotation	underpass_id	identificatie_left	car_accessible	pass_through	identificatie_right	b3_pand_deel_id	labels
11461	92139.81106793730	439536.05514660600	92165.22394669530	440075.03633848600	92814.86250525320	439417.95254586400	92890.85119572840	440201.4070365320	0.03164236373693330	E	0.0023083936673819800	-0.7069395246541520	-0.7072702311001350	0.9978740106172640	0.04769434805634310	-0.04441517869036310	0.06513163788277460	-0.7056640543808660	0.7055466760615240	0.0023083936673819800	-0.7069395246541520	-0.7072702311001350	0.9978740106172640	0.04769434805634310	-0.04441517869036310	0.06513163788277460	-0.7056640543808660	0.7055466760615240	199	199	199001111_0036_01_0162_P00_01.jpg	0.225418996078431	108.427248422948	108.791000366211	0.0037599999923259	0.219126	7141.23508467977	7102	5311.04073777729	5326	0.0626838322210643	0	-0.109754054511856	0	-0.118169809115977	0	1.63271641851488E-05	0	3.69526280672762E-05	0	9321.73238315893	91522.7252	807.845889421998	439742.2731	912.48077170438	912.471923828125	3.60209856658457	3.60899996757507	-45.0132456273457	-45.0068016052246	89.8129108079094	179.8296	-89.8129108079094		45.0132456273457		-3.60209856658457		\\192.168.0.74\SkylineUK\Projects\Kavel10\1578_102024081_Rotterdam AT Obliek\Images\PDC280\199001111_0036_01_0162_P00_01.jpg	14204	10652	IGI_PDC280_230227	Oblique_Back	Oblique	14204	10652	53.40704	40.05152	108.791	199	0	0	90	146464	NL.IMBAG.Pand.0599100000605676	FALSE	0	NL.IMBAG.Pand.0599100000702439	0	(19:0,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,1)
'''

image_id,width,height,fx,fy,cx,cy,X,Y,Z,omega,phi,kappa
199001111_0036_01_0162_P00_01.jpg,14204,10652.0,28933.776752194568,28933.776752194568,7102,5326,91522.7252,439742.2731,912.471923828125,206.78046640490263,-45.00680160522461,10303.477111525788


<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\S'
/var/folders/mg/1s263z_10vxf8w1d_74gbpxr0000gn/T/ipykernel_7679/3240900105.py:5: SyntaxWarning: invalid escape sequence '\S'
  11461	92139.81106793730	439536.05514660600	92165.22394669530	440075.03633848600	92814.86250525320	439417.95254586400	92890.85119572840	440201.4070365320	0.03164236373693330	E	0.0023083936673819800	-0.7069395246541520	-0.7072702311001350	0.9978740106172640	0.04769434805634310	-0.04441517869036310	0.06513163788277460	-0.7056640543808660	0.7055466760615240	0.0023083936673819800	-0.7069395246541520	-0.7072702311001350	0.9978740106172640	0.04769434805634310	-0.04441517869036310	0.06513163788277460	-0.7056640543808660	0.7055466760615240	199	199	199001111_0036_01_0162_P00_01.jpg	0.225418996078431	108.427248422948	108.791000366211	0.0037599999923259	0.219126	7141.23508467977	7102	5311.04073777729	5326	0.0626838322210643	0	-0.109754054511856	0	-0.118169809115977	0	1.63271

'\nParameters and intrinsic and extrinsic parameters\nName Input: 199001111_0036_01_0162_P00_01.jpg\nid\tULx\tULy\tURx\tURy\tLLx\tLLy\tLRx\tLRy\tGSD\tShotOrient\tInpho_11\tInpho_12\tInpho_13\tInpho_21\tInpho_22\tInpho_23\tInpho_31\tInpho_32\tInpho_33\tMatrix_11\tMatrix_12\tMatrix_13\tMatrix_21\tMatrix_22\tMatrix_23\tMatrix_31\tMatrix_32\tMatrix_33\tCAM\tCollection Input\tName Input\tMean median error Calculated\tFocal Length Calculated\tFocal Length Input\tPixel Size Input\tMedian Error Calculated\tPPoint X Calculated\tPPoint X Input\tPPoint Y Calculated\tPPoint Y Input\tK1 Calculated\tK1 Input\tK2 Calculated\tK2 Input\tK3 Calculated\tK3 Input\tP1 Calculated\tP1 Input\tP2 Calculated\tP2 Input\tX Calculated\tX Input\tY Calculated\tY Input\tAltitude Calculated\tAltitude Input\tOmega Calculated\tOmega Input\tPhi Calculated\tPhi Input\tKappa Calculated\tKappa Input\tYaw Calculated\tYaw Input\tPitch Calculated\tPitch Input\tRoll Calculated\tRoll Input\tPath Input\tWidth Input\tHeight Input\

In [1]:
'''
Made TIFF images
overlay on 

intersection3d tile and underpassto ge critical edge
'''
import math
def intrinsic_para(focal_length_mm, pixel_size_mm, ppx_px, ppy_px):
    # Convert focal length from mm to pixel units
    fx = focal_length_mm / pixel_size_mm
    fy = focal_length_mm / pixel_size_mm
    
    K = np.array([
        [fx,  0.0, ppx_px],
        [0.0, fy,  ppy_px],
        [0.0, 0.0, 1.0   ]], dtype=np.float64)
    
    return K

def extrinsic_para(omega, kappa, phi, x, y, z):

    r_omega = np.array([
        [1,  0, 0],
        [0, math.cos(omega),  -math.sin(omega)],
        [0, math.sin(omega), math.cos(omega)  ]], dtype=np.float64)

    r_phi = np.array([
        [math.cos(phi),  0, math.sin(phi)],
        [0, 1,  0],
        [-math.sin(phi), 0, math.cos(phi)  ]], dtype=np.float64)

    r_kappa = np.array([
        [math.cos(kappa), -math.sin(kappa), 0],
        [math.sin(kappa), math.cos(kappa),  0],
        [0, 0, 1  ]], dtype=np.float64)

    R = r_kappa @ r_phi @ r_omega
    x = 91522.7252
    y = 439742.2731
    z = 912.471923828125
    t = -R @ np.array([[x], [y], [z]]).astype(np.float32).reshape(3, 1)

    return R, t

'''
Additional information: 
yaw (-89.8129108079094)
pitch (45.0132456273457)
roll (-3.60209856658457)
X   9321.732383
Y   807.845889
Z  912.480772 

'''
# 199001111_0036_01_0162_P00_01.jpg
focal_length_mm = 108.427248
pixel_size_mm = 0.003760
ppx_px = 7141.235085
ppy_px = 5311.040738
K = intrinsic_para(focal_length_mm, pixel_size_mm, ppx_px, ppy_px)

omega = 3.602099
kappa = -45.013246
phi = 89.812911
x =  9321.732383
y =  807.845889
z = 912.480772 
R, t = extrinsic_para(omega, kappa, phi, x, y, z)



NameError: name 'np' is not defined

In [7]:
import pandas as pd
import geopandas as gpd

# OVERWRITE the kappa to original file 
csv_path = '/Users/belinaileen/Desktop/underpass-detection-experiments/3DBAG_underpass_heights/data/rotterdam_camera_parameters.csv'
geojson_path = '/Users/belinaileen/Desktop/rott_footprint2.geojson'

df_csv = pd.read_csv(csv_path)
gdf = gpd.read_file(geojson_path)

# 2. Create a mapping from 'Name Input' to 'Kappa Calculated'
# Drop geometry to work purely with tabular properties
df_geo = pd.DataFrame(gdf.drop(columns='geometry'))
kappa_mapping = df_geo.set_index('Name Input')['Kappa Calculated']

# 3. Overwrite the 'kappa' column
# This maps image_id to the new kappa value, keeping the original value if no match is found
df_csv['kappa'] = df_csv['image_id'].map(kappa_mapping).combine_first(df_csv['kappa'])

# 4. Save the updated CSV back out
df_csv.to_csv(csv_path, index=False)
print(f"Successfully updated '{csv_path}' with values from GeoJSON.")

Successfully updated '/Users/belinaileen/Desktop/underpass-detection-experiments/3DBAG_underpass_heights/data/rotterdam_camera_parameters.csv' with values from GeoJSON.


In [2]:
import pandas as pd
import geopandas as gpd

# OVERWRITE the omega to original file 
csv_path = '/Users/belinaileen/Desktop/underpass-detection-experiments/3DBAG_underpass_heights/data/rotterdam_camera_parameters.csv'
geojson_path = '/Users/belinaileen/Desktop/rott_footprint2.geojson'

df_csv = pd.read_csv(csv_path)
gdf = gpd.read_file(geojson_path)

# 2. Create a mapping from 'Name Input' to 'Kappa Calculated'
# Drop geometry to work purely with tabular properties
df_geo = pd.DataFrame(gdf.drop(columns='geometry'))
kappa_mapping = df_geo.set_index('Name Input')['Phi Calculated']

# 3. Overwrite the 'kappa' column
# This maps image_id to the new kappa value, keeping the original value if no match is found
df_csv['phi'] = df_csv['image_id'].map(kappa_mapping).combine_first(df_csv['phi'])

# 4. Save the updated CSV back out
df_csv.to_csv(csv_path, index=False)
print(f"Successfully updated '{csv_path}' with values from GeoJSON.")

Successfully updated '/Users/belinaileen/Desktop/underpass-detection-experiments/3DBAG_underpass_heights/data/rotterdam_camera_parameters.csv' with values from GeoJSON.
